In [2]:
import pandas as pd

In [3]:
from pathlib import Path
import os

In [4]:
import random

In [5]:
import numpy as np
import numpy as np

In [6]:

ROOT = Path.cwd()
DATA_DIR = ROOT / "Data"

In [7]:
df = pd.read_excel(DATA_DIR/"Well8_Assessment_RawResponses.xlsx")

In [8]:
scale_1_5 = 5
scale_1_10 = 10

In [9]:
reverse_columns = [
    "Physical_Q3_REV",
    "Mental_Q2_REV",
    "Mental_Q3_REV",
    "Safety_Q2_REV",
    "Connection_Q2_REV",
    "Inclusivity_Q2_REV"
]

In [10]:
for col in reverse_columns:
    max_scale = scale_1_5
    df[col] = max_scale + 1 - df[col]


In [11]:
dimensions = {
    "Purpose": {
        "questions": ["Purpose_Q1", "Purpose_Q2", "Purpose_Q3"],
        "scales": [scale_1_5, scale_1_5, scale_1_5]
    },
    "Physical": {
        "questions": ["Physical_Q1", "Physical_Q2", "Physical_Q3_REV"],
        "scales": [scale_1_10, scale_1_5, scale_1_5]
    },
    "Mental": {
        "questions": ["Mental_Q1", "Mental_Q2_REV", "Mental_Q3_REV"],
        "scales": [scale_1_10, scale_1_5, scale_1_5]
    },
    "Safety": {
        "questions" : ["Safety_Q1", "Safety_Q2_REV", "Safety_Q3"],
        "scales": [scale_1_10, scale_1_5, scale_1_5] 
    },
    "Connection": {
        "questions" : ["Connection_Q1", "Connection_Q2_REV", "Connection_Q3"],
        "scales": [scale_1_5, scale_1_5, scale_1_5]
    },
    "Resilience": {
        "questions" : ["Resilience_Q1", "Resilience_Q2", "Resilience_Q3"],
        "scales": [scale_1_5, scale_1_5, scale_1_5]
    },
    "Inclusivity": {
        "questions" : ["Inclusivity_Q1", "Inclusivity_Q2_REV", "Inclusivity_Q3"],
        "scales": [scale_1_5, scale_1_5, scale_1_5]
    },
    "Philosophy": {
        "questions" : ["Philosophy_Q1", "Philosophy_Q2", "Philosophy_Q3"],
        "scales": [scale_1_5, scale_1_5, scale_1_5]
    }
}

In [12]:
def normalize(value, max_scale):
    return (value/max_scale) * 50

In [13]:
def compute_dimension_score(row, questions, scales):
    score = []
    for q, max_scale in zip(questions, scales):
        score.append(normalize(row[q], max_scale))
    return sum(score) / len(score) if score else 0

In [14]:
for dim, config in dimensions.items():
    df[dim + "_Score"] = df.apply(
        lambda row: compute_dimension_score(row, config["questions"], config["scales"]),axis = 1
    )

In [15]:
dimension_score_columns = [dim + "_Score" for dim in dimensions.keys()]
df["WS_Score"] = df[dimension_score_columns].sum(axis=1)

In [16]:
def risk_flag(row):
    flags = []

    # Critical dimension
    for dim in dimensions.keys():
        if row[dim + "_Score"] < 15:
            flags.append(f"CRITICAL_{dim}")

    # Mental high risk
    if row["Mental_Score"] < 20:
        flags.append("HIGH_MENTAL_RISK")

    # Safety risk
    if row["Safety_Score"] < 20:
        flags.append("SAFETY_RISK")

    # Overall risk
    if row["WS_Score"] < 200:
        flags.append("OVERALL_HIGH_RISK")

    if not flags:
        return "LOW_RISK"

    return "; ".join(flags)

df["Risk_Flags"] = df.apply(risk_flag, axis=1)

In [17]:
df.to_excel("Data\Well8_Scored_Output.xlsx", index=False) # Access the file for the final results

print("Scoring complete.")
print(df[["student_id", "WS_Score", "Risk_Flags"]]) 

Scoring complete.
   student_id    WS_Score                                         Risk_Flags
0         S01  341.666667                                           LOW_RISK
1         S02  268.333333                                           LOW_RISK
2         S03  340.000000                                           LOW_RISK
3         S04  243.333333                                           LOW_RISK
4         S05  183.333333                                  OVERALL_HIGH_RISK
5         S06  155.000000                HIGH_MENTAL_RISK; OVERALL_HIGH_RISK
6         S07  341.666667                                           LOW_RISK
7         S08  268.333333                                           LOW_RISK
8         S09  253.333333                                           LOW_RISK
9         S10  183.333333                                  OVERALL_HIGH_RISK
10        S11  341.666667                                           LOW_RISK
11        S12  113.333333  CRITICAL_Purpose; CRITICAL_Conn

<>:1: SyntaxWarning: invalid escape sequence '\W'
<>:1: SyntaxWarning: invalid escape sequence '\W'
C:\Users\DEVIKA\AppData\Local\Temp\ipykernel_25856\6611864.py:1: SyntaxWarning: invalid escape sequence '\W'
  df.to_excel("Data\Well8_Scored_Output.xlsx", index=False) # Access the file for the final results


Numerical Personas Simulated

In [18]:
personas = {
    "S01": {  # Thriving Balanced
        "Mental": 0.80,
        "Physical": 0.85,
        "Connection": 0.80,
        "Purpose": 0.75,
        "Resilience": 0.80,
        "Safety": 0.90,
        "Inclusivity": 0.80,
        "Philosophy": 0.70
    },
    "S02": {  # High Achiever Anxiety
        "Mental": 0.35,
        "Physical": 0.50,
        "Connection": 0.55,
        "Purpose": 0.75,
        "Resilience": 0.45,
        "Safety": 0.90,
        "Inclusivity": 0.10,
        "Philosophy": 0.70
    },
    "S03": {  # Socially Isolated
        "Mental": 0.45,
        "Physical": 0.60,
        "Connection": 0.25,
        "Purpose": 0.50,
        "Resilience": 0.35,
        "Safety": 0.80,
        "Inclusivity": 0.14,
        "Philosophy": 0.55
    }
}

In [19]:
rotation_schedule = [
    ["Mental", "Physical"],
    ["Connection", "Purpose"],
    ["Physical", "Resilience"],
    ["Mental", "Connection"],
    ["Purpose", "Physical"],
    ["Mental"],
    ["Fun"],
    ["Inclusivity", "Mental"],
    ["Safety", "Connection"],
    ["Resilience", "Purpose"],
    ["Physical", "Inclusivity"],
    ["Mental", "Philosophy"],
    ["Mental"],
    ["Fun"]
]

In [20]:
def generate_mood_weights(mental_bias):
    """
    Creates a probability distribution across 5 emoji levels
    based on mental_bias (0–1).
    """

    # Map emoji positions to 0–1 scale
    positions = np.linspace(1, 0, 5) # [1.0, 0.75, 0.5, 0.25, 0.0]

    # Gaussian-like weighting centered at mental_bias
    weights = np.exp(-((positions - mental_bias) ** 2) / 0.05) 

    weights = weights / weights.sum() # For mental_bias = 0.8, 
                                      # [0.287, 0.607, 0.105, 0.002, 0.000]

    return weights.tolist()

In [21]:
weights_0_8 = generate_mood_weights(0.8)
print(weights_0_8)
print(sum(weights_0_8))

weights_0_3 = generate_mood_weights(0.3)
print(weights_0_3)
print(sum(weights_0_3))



[0.2865220223710657, 0.606567126119443, 0.10540556147317906, 0.001503529584162826, 1.7604521493528787e-06]
0.9999999999999999
[3.502202365397165e-05, 0.011003592736382543, 0.28378639698166225, 0.6007758071246302, 0.10439918113367101]
1.0


In [22]:
def simulate_card_response(bias, noise_level=0.15):
    noise = random.uniform(-noise_level, noise_level)
    value = bias + noise
    value = max(0, min(1, value))

    # Convert 0–1 → 1–5 scale
    return round(value * 4 + 1)

In [23]:
results = []

for student_id, biases in personas.items():

    dimension_scores = {dim: [] for dim in dimensions.keys()}

    for day in rotation_schedule:

        # Mood influenced by mental bias and random noise
        mental_bias = biases["Mental"]
        mood_weights = generate_mood_weights(mental_bias)
        mood_index = np.random.choice(range(5), p=mood_weights)

        emoji_scores = [5, 4, 3, 2, 1]
        mood_score = emoji_scores[mood_index]
        dimension_scores["Mental"].append(mood_score)
        
        for dim in day:
            if dim == "Fun":
                continue

            bias = biases.get(dim, 0.5)
            score = simulate_card_response(bias)
            dimension_scores[dim].append(score)

    # ----- Normalize to 0–50 -----
    normalized = {}
    normalized["student_id"] = student_id
    for dim, values in dimension_scores.items():
        avg = sum(values) / len(values)
        normalized[dim + "_Score"] = round((avg / 5) * 50, 2)

   
    results.append(normalized)

baseline_df = pd.DataFrame(results)

dimension_cols = [col for col in baseline_df.columns if "_Score" in col]
baseline_df["WS_Score"] = baseline_df[dimension_cols].sum(axis=1)

baseline_df["Risk_Flags"] = baseline_df.apply(risk_flag, axis=1)

print(baseline_df)





  student_id  Purpose_Score  Physical_Score  Mental_Score  Safety_Score  \
0        S01          36.67            47.5          42.0          50.0   
1        S02          40.00            27.5          24.0          50.0   
2        S03          33.33            32.5          27.0          50.0   

   Connection_Score  Resilience_Score  Inclusivity_Score  Philosophy_Score  \
0              40.0              40.0               40.0              30.0   
1              30.0              25.0               10.0              30.0   
2              20.0              20.0               20.0              30.0   

   WS_Score            Risk_Flags  
0    326.17              LOW_RISK  
1    236.50  CRITICAL_Inclusivity  
2    232.83              LOW_RISK  


Chatbot 

In [24]:
student_profile = baseline_df.iloc[0].to_dict()
print(student_profile)

{'student_id': 'S01', 'Purpose_Score': 36.67, 'Physical_Score': 47.5, 'Mental_Score': 42.0, 'Safety_Score': 50.0, 'Connection_Score': 40.0, 'Resilience_Score': 40.0, 'Inclusivity_Score': 40.0, 'Philosophy_Score': 30.0, 'WS_Score': 326.17, 'Risk_Flags': 'LOW_RISK'}


In [ ]:
class WellBeingBot:
    def __init__(self, student_profile):
        self.student_profile = student_profile['student_id']
        self.dimension_scores = {dim: student_profile[dim + "_Score"] for dim in dimensions.keys()}

    def greet(self):
        print(f"\nHi {self.student_profile}!")
        print("\nReady for a quick check_in ?\n")

    def mood_check(self):
        print("\nHow are you feeling right now?\n")
        print("1. 😄 Great")
        print("2. 😐 Okay")
        print("3. 😐 Meh")
        print("4. 😟 Not good")
        print("5. 😢 Struggling")

        choice = int(input("\nChoose 1-5:"))

        emoji_scores = [5, 4, 3, 2, 1]
        mood_score = emoji_scores[choice - 1]

        self.update_dimension("Mental", mood_score)

        return mood_score # Could be used to formulate the dimension questions
    
    def dimension_card(self, dimension):

        print(f"\nQuick question about {dimension}.")

        print("""\n1. Yes
              2. A little
              3. Not really\n""")

        choice = int(input("Choose 1-3: "))

        mapping = {1:5, 2:3, 3:1}
        score = mapping.get(choice,3)

        self.update_dimension(dimension, score)


    def update_dimension(self, dimension, new_score):

        old_score = self.dimension_scores[f"{dimension}_Score"]

        normalized = (new_score - 1 / 4) * 50

        updated_score = (0.7 * old_score) + (0.3 * normalized)

        self.dimension_scores[f"{dimension}_Score"] = round(updated_score, 2)

    
    def evaluate_risk(self):

        

        for dim in dimension_scores.keys():
            if self.dimension_scores[dim] < 15:
                print(f"⚠️  CRITICAL_{dim}!")
            if sum(self.dimension_scores.values()) < 200:
                print("⚠️  OVERALL_HIGH_RISK!")

        


    def run_daily_checkin(self):
        self.greet()
        self.mood_check()

        dimension_card = rotation_schedule[0][0] # For demo, we just take the first day’s dimensions
        self.dimension_card(dimension_card)

        risk_flags = self.evaluate_risk()
        ws = 



In [31]:
bot = WellBeingBot(student_profile)
bot.run_daily_checkin()


Hi S01!

Ready for a quick check_in ?


How are you feeling right now?

1. 😄 Great
2. 😐 Okay
3. 😐 Meh
4. 😟 Not good
5. 😢 Struggling


In [ ]:
class WellbeingBot:

    def __init__(self, student_profile):
        """
        student_profile: dictionary containing student baseline scores
        """
        self.student_id = student_profile["student_id"]

        # Store current wellbeing scores
        self.dimension_scores = {
            "Mental_Score": student_profile["Mental_Score"],
            "Physical_Score": student_profile["Physical_Score"],
            "Connection_Score": student_profile["Connection_Score"],
            "Purpose_Score": student_profile["Purpose_Score"],
            "Resilience_Score": student_profile["Resilience_Score"],
            "Safety_Score": student_profile["Safety_Score"],
            "Inclusivity_Score": student_profile["Inclusivity_Score"],
            "Philosophy_Score": student_profile["Philosophy_Score"]
        }

    # ---------------------------------
    # Greeting
    # ---------------------------------
    def greet(self):
        print(f"\nHi {self.student_id}! 👋")
        print("Ready for a quick 30-second check-in?\n")

    # ---------------------------------
    # Mood Check
    # ---------------------------------
    def mood_check(self):

        print("How are you feeling right now?")
        print("1 😄  Great")
        print("2 🙂  Okay")
        print("3 😐  Meh")
        print("4 😟  Not great")
        print("5 😢  Really struggling")

        choice = int(input("Choose 1–5: "))

        emoji_scores = [5,4,3,2,1]
        mood_score = emoji_scores[choice-1]

        self.update_dimension("Mental", mood_score)

        return mood_score

    # ---------------------------------
    # Dimension Card
    # ---------------------------------
    def dimension_card(self, dimension):

        print(f"\nQuick question about {dimension}.")

        print("1 Yes")
        print("2 A little")
        print("3 Not really")

        choice = int(input("Choose 1–3: "))

        mapping = {1:5, 2:3, 3:1}
        score = mapping.get(choice,3)

        self.update_dimension(dimension, score)

    # ---------------------------------
    # Update Dimension Score
    # ---------------------------------
    def update_dimension(self, dimension, new_score):

        old_score = self.dimension_scores[f"{dimension}_Score"]

        # Convert 1–5 scale to 0–50
        normalized = ((new_score - 1) / 4) * 50

        # Smooth update (retain baseline influence)
        updated_score = (0.7 * old_score) + (0.3 * normalized)

        self.dimension_scores[f"{dimension}_Score"] = round(updated_score,2)

    # ---------------------------------
    # Risk Evaluation
    # ---------------------------------
    def evaluate_risk(self):

        mental = self.dimension_scores["Mental_Score"]
        safety = self.dimension_scores["Safety_Score"]

        flags = []

        if mental < 20:
            flags.append("HIGH_MENTAL_RISK")

        if safety < 15:
            flags.append("SAFETY_RISK")

        if len(flags) == 0:
            flags.append("LOW_RISK")

        return flags

    # ---------------------------------
    # Compute WS Score
    # ---------------------------------
    def compute_ws(self):

        ws = sum(self.dimension_scores.values())
        return round(ws,2)

    # ---------------------------------
    # Run Full Daily Check-in
    # ---------------------------------
    def run_daily_checkin(self):

        self.greet()

        mood = self.mood_check()

        # Example rotating card
        self.dimension_card("Connection")

        risk_flags = self.evaluate_risk()
        ws = self.compute_ws()

        print("\nUpdated Scores:")
        print(self.dimension_scores)

        print("\nWell8 Score:", ws)
        print("Risk Flags:", risk_flags)

        print("\nThanks for checking in today ⭐")